# Postgres Query Optimization

## Why Optimize

* SQL is a **declarative language**, meaning we describe the result we want to obtain but don't specify how the result should be obtained.
* An **imperative language** by contrast means we specify the exact steps to be taken to obtain the desired result.

## Theory: Yes We Need It

### Query Processing Overview
* Compile and transform a SQL statement into an expression consisting of high-level logical operations, known as a logical plan.
    * The source code is parsed and an internal representation is generated. 
* Optimize the logical plan and convert it into an execution plan.
    * The optimizer performs two kinds of transformations: it replaces logical operations with their execution algorithms and possibly changes the logical expression structure by changing the order in which logical operations will be executed.
    * The optimizer tries to find a logical plan and physical operations that minimize required resources, including execution time.
    * The output of the optimizer is an expression containing physical operations. This expression is called a (physical) execution plan.
    * The PostgreSQL optimizer is called the query planner.
* Execute (interpret) the plan and return results.
    * The query execution plan is interpreted by the query execution engine, frequently referred to as the executor in the PostgreSQL community, and output is returned to the client application.
 
### Relational Operations
* A relation can be thought of as a table that with relational operations produces another relation or table.

* A *filter* operation applies restrictions (where conditions) on the relation to produce another relation after these restrictions have been applied.
* A *project* operation concerns selection of columns needed and optionally removes duplicates.
* A *product* opertaion is the cartesian product of two relations giving all the possible combinations of these relations.

### Equivalence Operations
* **Commutativity** – JOIN(R,S) = JOIN (S,R)
* **Associativity** – JOIN(R, JOIN(S,T) = JOIN(JOIN(R,S), T)
* **Distributivity** – JOIN(R, UNION(S,T)) = UNION(JOIN(R,S), JOIN(R, T))

## Even More Theory

* Database objects are divided into blocks of the same length
* A block is the unit that is transferred between the hard drive and the main memory, and the number of I/O operations needed to execute any data access is equal to the number of blocks that are being read or written.
* Table rows are stored using a data structure called a heap. Other objects (e.g., indexes) may use blocks differently.

### Two ways of using indexes
* Index scan
* *Bitmap index scan* followed by a *bitmap heap scan*
    * Will record if a particular block contains information needed by an index
    * Only when a block is recorded to have information by all indexes needed for retrieval will this block be used.
    * After the final candidate list is computed, the candidate blocks are read sequentially using a bitmap heap scan (a heap scan based on a bitmap), and for each block, the individual records are examined to recheck the search conditions

### Types of Scans
* Full Table Scan
    * Scanning the entire table for a value 
* Index-Only Scans
    * If all values of a query are contained in the index than there is no need to go to the table to retrieve data. Will always be less than a full table scan since less data and goes up logaritmically.
* Index Access
    *  Index Access goes up logarithmically and crosses the threshold of the constant cost of a full table scan.
        * When the index is very selective it is prefereable to use Index Access but when it isn't a full table scan is preferred since the cost of accessing the index becomes just as costly as the full scan.
     
### B-Tree Indexes
* Supports Equality, greater than, less than and the Between operator (range search)
* A large number of deletions may result in too large of a b-tree depth and degrade the performance. To avoid the degradation, the index should be rebuilt

### Hash Index
* This uses a hash function to calculate the address of an index block with the index key. This type of index has better performance than a B-tree index for equality conditions. However, this index is completely useless for range queries.

### Combinining Relations
* A Cartesian product is the the product of nesting a loop of all relations of one table into a loop of all relations of another joined table.
    * If two tables are combined on an indexed value this can be very fast
* Hash-based algorithm - One table has its values placed into various buckets that the second table can find via hash and then scan for matching values
* Sort-Merge Algorith - If both tables are sorted there is no need to do a complete scan of the table as soon as last value is found in sorted order you don't go further.

## Understanding Execution Plans

* The output of the PostgreSQL optimizer is an execution plan.
* The job of the optimizer is to build the best possible physical plan that implements a given logical plan
* The optimizer replaces logical operations with corresponding physical execution algorithms and (possibly) changes the logical expression structure by changing the order in which logical operations are executed.

Steps during Optimiztion
* Query Re-Write - Like substituting views with their textual representation.
* Determining the possible orders of operations
* Determining the possible execution algorithms for each operation
* Comparing the costs of different plans
* Selecting the optimal execution plan

Postgres will find the optimal algorith for a sub-plan and then build subsequent plans off of this one.

* If too many joins are part of a query the optimizer will stop short of finding the best join sequence for all of these joins and use heuristics instead.
* This threshold is called the *geqo_threshold* configuration parameter.

Cost of an execution plan depends on:
* Cost formulas of algorithms used in the plan
* Statistical data on tables and indexes, including distribution of values
    * If value selecting on is present thoughout most of the table it would likely not be of value to use an index
    * **ANALYZE** will recalculate statistics for a table. Important to keep updated.
* System settings (parameters and preferences), such as join_collapse_limit or cpu_index_tuple_cost

## Short Queries and Indexes

* A short query attempts to select a fraction of the data held in tables
* Attempt to avoid large intermediate results
* The optimization goal is to reduce the size of the result set as early as possible.
    * The desired join order is the one that would prompt the usage of indexes with lower selectivity first.
* To make the biggest benefit the most selective criteria should be indexed
* Foreign keys don't have indexes that are automatically created. If they have enough distinct values then an index should be created.

### Column Transformations
* When using a column transformation such as lower(last_name) important to not that an index on last_name will not be used but an index on lower(last_name) will be used.
    * An ANALYZE is useful after creating an expression index like lower(last_name)
 
### The like operator
* A B-tree index does not cover the like operator
    * **where last_name like 'johns%'**
* Create a pattern search index for the like operator
    * `CREATE INDEX account_last_name_lower_pattern ON account (lower(last_name) text_pattern_ops);`
 
### Using multiple indexes
* A bitmap can be used that identifies when an index is present within a block and then ANDing or ORing with other bitmaps representing indexes to see if criteria is satisfied.

### Compound Indexes
* `CREATE INDEX flight_depart_arr_sched_dep ON  flight(departure_airport,arrival_airport,scheduled_departure)`
* The first column must be part of the search criteria otherwise the index is not used. Not necessary to have a separate index on departure_airport now.
* Allows for a greater amount of selectivity of rows

### Covering Indexes
* Used to support index-only scans so that index will also include in the index Select values needed so that all data can be found in the index.
* Began usage in Postgres 11
* `CREATE INDEX flight_depart_arr_sched_dep_inc_sched_arr ON flight(departure_airport,arrival_airport,scheduled_departure) INCLUDE (scheduled_arrival);`

### Excessive Selection Criteria
* May be needed extra Selection criteria along with current criteria in an attempt to capture the data needed but minimize results selected

### Partial Indexes
* Index created on subset of table using a where condition
* `CREATE INDEX flight_actual_departure_not_null ON flight(actual_departure) WHERE actual_departure IS NOT NULL`
* If not that many values are present for the where condition it would be of value to quickly find these rarer cases of interest.
* There where condition would have to be used in order for the index to be used

### Avoiding/Ignoring Index Usage
* It may be beneficial to not use an index since would be faster to perform a sequential scan instead
* Potentially could force an index to not be used by using column transformations
* Postgres itself may ignore index usage since the amount of rows that will be selected are very large
    * Ordering can be an issue if column ordered upon is not indexed since will need to find all these values and sort on them
 
### When to build Indexes
* Use partial and covering indexes when it makes sense
* Too many indexes can lead to slow inserts/updates so good to monitor
* The unique/primary key indexes and foreign keys that reference other unique/primary key fields are usual culprits of slowness, as well as triggers on insert/update
* You should monitor excessive growth of index size

### Indexes not Needed
* Should review for indexes that aren't used by monitoring *pg_stat_all_indexes*

## Long Queries and Full Scans

* A query is considered long when a large amount of rows from one or more tables is needed to calculate the result.

Optimization Strategies of Long Queries
* Avoid multiple table scans
* Reduce the size of the result as early as possible

* For a long query a table scan is preferred since will perform better than an index given the size. We want to ensure an index is not used in this case.

### Long Queries and Hash Joins
* Hash Joins are preferable for long queries as cost is less for hash as opposed to nested loop joins when the size of the data is large
* Hash joins work best when the first argument fits into main memory. The size of memory available can be tuned with server parameters.
* A merge join may also be used but can be more efficient when at least one of the tables is presorted.
    * May be of value in loading data in buld to sort the data first. There is also the cluster option.

### Long Queries and the Order of Joins
* The most restrictive joins that reduce the data selected should be executed first.
* A Semi-Join return rows from one table where there are rows matching in a subsequent table
    * Can be written using either the *Exists* or *IN* operator
    * May be executed as a regular join for execution plan depending on cardinality and filter selectivity. Also may use join if there is no need to remove duplicates.
* An Anti-Join returns rows from one table where there are no rows matching on the match criteria against a subsequent table.
    * Can be written using either the *NOT Exists* or *NOT IN* operator
* The *join_collapse_limit* caps the number of tables in a join that will be still processed by the cost-based optimizer. The default value for this parameter is 8.
    * If the number of tables is greater than 8 then the query will simply execute the joins in the order the tables are listed in the SELECT statement.
    * It is not a good idea to set this value too high as the number of plans to consider increases by n!
    * Table statistics are not available for intermediate results which may cause the optimizer to choose a suboptimal join order
        * It may be beneficial to force the desired join order by setting *join_collapse_limit* to 1 and have a plan in which the joins will be executed in the order they appear in the SELECT statement.
     
### Grouping: Filter First, Group Last
* Filtering should be pushed inside the grouping so that we only group on the needed data
* Filter rows are not needed for an aggregate prior to grouping.

### Grouping: Group First, Select Last
* In some cases grouping shound occur as early as possible when grouping will reduce the size of the intermedicate dataset.
  
## Using Set Operations
* For large queries using SET operations may prompt the optimizer to choose more efficient algorithms and improve readability.
    * Use EXCEPT instead of NOT EXISTS and NOT IN.
    * Use INTERSECT instead of EXISTS and IN.
    * Use UNION instead of complex selection criteria with OR.
* For both hash joins and set theoretical operations, if the participating datasets can’t fit into main memory, the execution speed increases significantly.

## Avoiding Multiple Scans
* Pull values from an EAV table (entity-attribute-value) into a subquery before joining to other tables. Inside the subquery we can use the technique of grouping before joining where we filter the needed attribute values of our grouping id.

```
JOIN (
SELECT
       cf.passenger_id,
       coalesce(max (custom_field_value )
               FILTER (WHERE custom_field_name ='passport_num' ),'')
                                                   AS passport_num,
        coalesce(max (custom_field_value )
               FILTER (WHERE custom_field_name ='passport_exp_date' ),'')
                                                   AS passport_exp_date,
        coalesce(max (custom_field_value )
               FILTER (WHERE custom_field_name ='passport_country' ),'')
                                                   AS passport_country
FROM custom_field cf
WHERE cf.passenger_id<5000000
GROUP BY 1
   ) info  USING (passenger_id)
```

## Conclusion
* Indexes do not necessarily make queries run faster and can make a long query run slower
* Long queries are optimized by reducing the size of intermediate results and doing the necessary work on as few rows as possible
    * This is accomplished by being mindful of join order, applying semi- and anti-joins, and filtering before grouping, grouping before joining, and applying set operations.

## Long Queries: Additional Techniques

### Temporary Tables

Issues with Temporary Tables
* **Indexes** - We aren't able to use indexes of the source tables and may need to create our own
* **Statistics** - Not able to use stat of source tables or may need to analyze the temp table
* **Disk space** - Temp tables are stored to disk which takes space
* **Excessive IO** - The tables are written to disk which takes extra time to read and write from disk

An imortant negative impact of creating a temp table is that it prevents the optimizer from doing rewrites. You are preventing the optimizer from choosing the optimal join order.

### Common Table Expressions (CTEs)

CTEs are useful when tables are used more that once or for use with recursion and INSERT/DELETE/UPDATE statements.

The tables involved in the CTE are not counted against *join_collapse_limit* property.

For all versions below 12 a CTE would materialize to main memory and process like a temporary table.

In Postgres 12 if a CTE is not used with recursion and is used only once then it will be inlined into the outer query, otherwise the old behavior will be preserved.
* One can force materialization by using the keyword *MATERIALIZED* or force inlining by using the keyword *NOT MATERIALIZED*
    * ``WITH flights_totals AS MATERIALIZED (...)``
 
### Views

A view is a database object that stores a query that defines a virtual table. It is a virtual table in the sense that we can call the view much like we can call a table but no data is stored except for the query used to define itself.

Views can be optimization fences that can prevent the optimizer from applying the correct join order when used with other tables.

The creation of a view also creates rules that restrict access to the underlying tables. It can be used as a security layer or to define a reporting entity. They don't have any performance benefits however.

### Materialized Views

A materialized view is a database object that combines both a query definition and a table to store the results of the query at the time it is run.

Can be refreshed concurrently so not locked for access
* ``REFRESH MATERIALIZED VIEW CONCURRENTLY flight_departure_mv``
    * Needs to have a unique index for concurrent refresh
 
### Partitioning

Currently, PostgreSQL supports the following partitioning methods: range, list, and hash partitioning. The most common case is range partitioning.

The way partitioning can improve performance is that if a query contains conditions on the partitioning key, the search is limited to these partitions only.

The values of the partitioning key should be known prior to the SQL statement execution. The latter means that this value can't be passed as a parameter or as a result of a subselect.

The partitioned table itself is a virtual table and does not store any rows. Instead, each table row is stored in one or more partition tables according to rules specified when the partitioned table is created.


```
CREATE TABLE boarding_pass_part (
     boarding_pass_id SERIAL,
     passenger_id BIGINT NOT NULL,
     booking_leg_id BIGINT NOT NULL,
     seat TEXT,
     boarding_time TIMESTAMPTZ,
     precheck BOOLEAN,
     update_ts TIMESTAMPTZ
)
PARTITION BY RANGE (boarding_time);
--create partitions
--
CREATE TABLE boarding_pass_may
PARTITION OF boarding_pass_part
FOR VALUES
FROM ('2023-05-01'::timestamptz)
TO ('2023-06-01'::timestamptz) ;
```

### Why Create a Partitioned Table?

* Partition tables can be easily added or dropped. Better to drop than delete rows followed by vacuum.
* Easy to ATTACH and DETACH partitions. You can detach the oldest partition and attach it to an archiving table.
* Partitioning may be used to distribute large amounts of data across several database servers as a partition can be a foreign table.
* Difficult to run autovacuum on a table when it reaches terabyte size

### Parallelism

* Starting in version 10, PostgreSQL has capability to enable parallel query execution.
* Two configuration parameters, **max_parallel_workers** and **max_parallel_workers_per_gather**, define how many parallel processes are available and how many can be spanned in one session.
* Parallel execution is beneficial for massive scans and hash joins. Both scans and hash joins are typical for long queries, for which the speed-up is usually most significant. Mostly beneficial when bulk data is processed.

## Optimizing Data Modification

* Data Modification or DML includes INSERT, UPDATE, DELETE statements and can be optimized at the point of optimizing selection for the DML and the actual modification.
    * Modications can cause blocking locks that slow the execution of other statements.
 * A Select needs to wait unitll all needed blocks are fetched into memory while a DML operation can immediately work on a block with writing to disk occurring asynchonously. They will be written at commit.
     * DML does take resources such as modifiying indexes and registering updates to the WAL (write-ahead log)
     * Writes consume harware resources and I/O bandwith
     * There is additional work with maintenance and and data restructuring as in VACUUM that can block access to the modified data.
 * With Update statements, the order of operations is crucial so some operations may be delayed or declined. Maintaining this order is the responsibility of the concurrency control or transaction processing subsystem.

### The Impact of Concurrency Control

* While modifying data a transaction will perform a lock on the data, if another transaction already has a conflicting lock then the execution is delayed until the lock is released. Lock waiting is the primary cause of delays in modification operations.

* Data modifications must be registered in WAL records on the hard drive before a transaction can commit. The WAL is written sequntially so the medium of hard drive such as SSD will impact performance. Too many commits such as one transaction for each update will impact performance. Holding on to a transaction for too long can also impact performance due to locking.
    * Using jdbctemplate update vs batchupdate. Should review if needing refactoring to greater usage of batchupdate.
 
Snapshot isolation concurrency
* Postgres doesn't update in place. Instead a newly allocated row is created for the new version of the item while the previous version is still accessible as is noted as a 'dead row'.
    * A **VACUUM** operation removes 'dead rows' and consolidates the free space in a block when old versions are no longer needed for currently running transactions.
* If another transaction has updated a particular row but did not commit before the start of the read operation, the read operation will return for the obsolete 'dead row' version. This imporves throughput since read operations don't need to wait.
* Concurrent writes of the same data is not allowed. Postgres utilizes the strategy that the first commit of competing transactions will get to write the data while other transactions will be aborted.
* Postgres uses write locks to enforce consistency, behavior after commit depends on the isolation level:
    * *REPEATABLE READ* isolation level guarantees that data read will be consistent on second read. If a second transaction modifies data before the first transaction gets a chance to complete, the first transaction will be aborted.
    * *REPEATABLE COMMIT* isolation level allows for committed data of other transactions to be read on subsequent reads of a transaction. A transaction will not be aborted and will read and move forward with the modified data committed by other transactions.

### Data Modification and Indexes

* Adding an extra index results in only a 1% increase in INSERT/UPDATE time.
* **CREATE INDEX** operation in PostgreSQL locks the table for all DML operations
* **CREATE INDEX CONCURRENTLY** takes longer to complete but allows DML to be executed.
* Postgres attempts to insert new version of update row into the same block, if update does not modify any indexed columns there is no need to modify any indexes.

### DML and Vacuum

* DELETE operation marks deleted rows as removed with UPDATE inserts an new version of the row and marks the previous version as outdated. As soon as these rows are not needed for active transactions, they become dead. As the number of dead rows grows, it effectively reduces the number of active rows in a block and thus slows down subsequent heap scans.
* The space occupied by dead rows (i.e., deleted tuples) remains unused until it is reclaimed by a VACUUM operation. Too much dead tuples cause table bloat, but this normally is addressed by the autovacuum daemon.
    * AUTOVACUUM system parameters can be tuned to ensure fast completion and adequate removal of dead tuples.
 
### Mass UPDATE/DELETE

* After mass update/delete SELECTs will generally be slower due to the large increase in dead tuples which will result in more blocks needing to be read to find active tuples.
* May need evaluate if consecutive updates on the same set of recrods makes sense or if it would be better to partition the table and replace mass updates with DROP/DETACH partions and subsequent CREATE/ATTACH partitions.

### Frequent Updates

* When a table experiences frequent updates that effects only a small number of rows it may benifit from a lower fill factor giving a greater amount of space to updated rows.
    * **fillfactor** is a storage parameter that defines the percentage of free space in table blocks. The default value of this parameter is 100 which means that the system should fit as many live rows of data as possible within each block. Free space normally apears after vaccuuming. 
* A smaller value of fill factor means an increased number of blocks are needed to store data and increases the number of reads during a heap scan. At the same time there is a greater likelihood to find updated rows within the same page, Heap Only Tuple (HOT), improving performance.
* A smaller fill factor makes it less likely that an index will split into a separate page and that table bloat will occur.
### Referential Integrity and Triggers
* The presence of multiple foreign keys can potentially slow DML as for each INSERT/UPDATE operation the table needs to check the integrity constraints. Triggers may also slow down DML for similar reasons.

## Design Matters

### Other Types of Databases

### Entity-Attribute-Value Model
* Each row describes an enitity, an attribute of that entity, and a value for that entity. This allows for greater flexibility but at the expense of performance and also strong checks of referential integrity.

### Key-Value Model
* Stores complex objects within a single field such as a JSON or JSONB object. This also makes it difficult to perfom type checks and institute referenctial integrity constraints.
    * A suggestion is using JSON when needed as a whole object and move attributes needed for search criteria to their own column in addition to storing them as components of a larger object. 

### Hierarchical Model
* Data is stored in a parent-child tree like structure. May be beneficial if data is static but may have issues in that doesn't provide flexibility and when data may fall into one or more parent nodes.

* Postgres can use a combination of the above where it makes sense.

### Must We Normalize

Informally, a database schema is normalized if all column values depend only on the table primary key, and data is decomposed into multiple tables to avoid repetition.

The primary purpose of normalization is not to improve performance. Normalization creates a clean logical structure and helps ensure data integrity, especially when it is supported by referential integrity constraints.

Normalization can improve perfomance when attempting to find distinct values of some attribute with high selectivity. Since less rows in the normalized table as opposed to the great amount of duplication in a denormalized table.

### Use and Misuse of Surrogate Keys

Surrogate keys are unique values generated by the system in order to identify objects stored in the database such as from a sequence.

Issue with using surrogate keys is that they have no relation to the data that is attempted to be saved and may hide duplication issues.

It may be better to use as a primary key a value or values in the data that should be unique.

A surrogate key may be used so that compound keys are not needed to be referenced and in situations where no natural primary key from the data can be found.

## What about Configuration Parameters?

### Memory Allocation

**shared_buffers** The amount of memory allocated for data read from the hard drive so that it can be worked upon. Resetting this parameter requires an instance restart. After subtracting this amount from Total RAM there should be enough memory left for **work_mem**.

* Recommendation: Start with 25% of RAM and increase it until cache_hit_ratio is close to 90%.
    * cache_hit_ratio: The success rate of finding data in cache

**effective_cache_size** An adivisory value used for planning purposes that tells the planner how much memory is available for caching data (includes memory cache, disk cache, filesystem cache). A higher value makes the planner choose to use indexes at a higher rate than sequential scans.
    
* Recommendation: 75% RAM is usually recommended but not required (and this number is not related to the shared_buffers value)

**max_connections** Sets the max number of concurrent connections that the system can handle simultaneously. Too many connections will take up resources even when idle as memory is allocated for each session and can lead to slow and un-responsive behavior.

* Recommendation: Start from several hundred, observe system behavior, and adjust accordingly.

**work_mem** Amount of memory used per query operation node for sorting data and hash tables. If more memory is needed the query will resort to swapping to disk and take longer to complete. This amount is often tied to *max_connections* ensuring that each connection has enough memory to work on data. A smaller mount of max_connections would thus lead to a greater amount for *work_mem*.

* Recommendation: Divide 25% of RAM with max_connections. Assuming that the host has only one database.
* Possible to set *work_mem* on a session level
    * ``set work_mem = '200GB';``
    * ``reset work_mem;``

**maintenance_work_mem** This parameter determines the maximum amount of memory used for maintenance operations like VACUUM, CREATE INDEX, ALTER TABLE, ADD FOREIGN KEY, and data-loading operations (e.g., COPY). 

### Tuning Parameters for Better Performance

* Benefical to run EXPLAIN (ANALYZE, BUFFERS) to analyze usage of buffer cache and see where we may be doing a lot of work in a query
    *  ``Buffers: shared hit=512 read=7822``
        * *shared hit* means the amount of blocks that has already been cached in main memory. It was not necessary to read this data from the hard disk.
        * *shared read* means the amount of blocks that had to be read from the hard disk.
* It is important to tune parameters to the system at hand but once done the most important elemnt in improving query performance is analysis of how you may restructure a query, cluster data and index creation where needed.

### Application Development and Performance

The way applications interact with the database matters. Ideas of encapsulation and isolation of business objects to database objects can cause poor performance when too many small queries are made to the database. 
* Attempts to improve performance by increasing memory, changing architecture, etc. may not address the root cause of the issue which is not addressing the mismatch of the needs of the application with the database.

Ways that Postgres can package data into a complex object so that multiple database calls are reduced to a single call if needed.
* Postgres itself is an object-relational database.
* Supports the creation of custom types.
* Functions can return sets, including sets of records.
* Supports JSON/JSONB data types


### Functions

Internal functions are written in the C language and are integrated with the PostgreSQL server.

User-Defined Functions
* Query language functions, that is, functions written in SQL
* C functions (written in C or C-like languages, like C++)
* Procedural language functions, written in a supported procedural language (referred to as PL)
    * The standard PostgreSQL distribution supports four procedural languages: PL/pgSQL, PL/Tcl, PL/Perl, and PL/Python. 

The body of a function is String that has been dollar-qouted as in (Example PL/pgSQL function):

```
$BODY$ 
    BEGIN
    ...
    END;
$BODY$
LANGUAGE plpgsql;
```

Custom function converting text to date capturing any exceptions that may occur

```
CREATE OR REPLACE FUNCTION text_to_date(input_text text)
  RETURNS date AS
$BODY$
BEGIN
     RETURN input_text::date;
EXCEPTION WHEN OTHERS THEN
  RETURN null::date;
END;
$BODY$
  LANGUAGE plpgsql;
```

When you create a function only cursory checks are made that it is working. You will need to execute the function and test any conditional paths to be assured it is working.
* No execution plan is saved. It is not stored compiled but rather as source code.
* No checks for existence of tables, columns, or other functions are performed.
* A prepared statement is only created when the execution path hits a specific command and is analyzed. Only statements executed with current conditional logic will be tested and prepared statement created.
    * Prepared statement will be re-used if function and conditional logic applies again in the same session.
 
Postgres functions are atomic:
* You can't initiate a transaction inside a function
* Optimizer doesn't know about function execution plan
    * May want to run the internal function SQL with parameter values to see this

 Using a function inside a Select list:

```
SELECT
       flight_id,
       num_passengers(flight_id) AS num_pass
FROM flight f
WHERE departure_airport='ORD'
      AND scheduled_departure BETWEEN '2023-07-05' AND '2023-07-13'
```

The above function will run as many times as the rows are selected. It will use a prepared statement, but the execution plan won't take into account different statistics between calls since this is cached in the same session. Would be better to simply calculate the count within the existing SQL.

Functions work as an optimization fence similar to views and CTEs.

### User-Defined Data Types

Simple user-defined types include the following categories: domain, enum, and range
* `CREATE DOMAIN timeperiod  AS tstzrange;`
* `CREATE TYPE mood AS ENUM ('sad', 'ok', 'happy’);`
* `CREATE TYPE mood_range AS RANGE...`

A composite type represents a row or record:

```
CREATE TYPE boarding_pass_record AS (
     boarding_pass_id int,
     booking_leg_id bigint,
     flight_no text,
     departure_airport text,
     arrival_airport text,
     last_name text,
     first_name text,
     seat text,
     boarding_time timestamptz
);

DECLARE
v_new_boarding_pass_record boarding_pass_record;
```

You can also also create a type who has as its member another user-defined composite type and return this type in your function.

```
CREATE OR REPLACE FUNCTION booking_leg_select (p_booking_leg_id int)
RETURNS SETOF booking_leg_record
AS
$body$
BEGIN
     RETURN QUERY
           SELECT
                bl.booking_leg_id,
                leg_num,
                bl.booking_id,
               (SELECT row(
                       flight_id,
                       flight_no,
                       departure_airport,
                       da.airport_name,
                       arrival_airport,
                       aa.airport_name ,
                       scheduled_departure,
                       scheduled_arrival)::flight_record
               FROM flight f
               JOIN airport da on da.airport_code=departure_airport
                     JOIN airport aa on aa.airport_code=arrival_airport
                     WHERE flight_id=bl.flight_id
                      ),
                   (SELECT array_agg (row(
                            pass_id,
                            bp.booking_leg_id,
                            flight_no,
                            departure_airport ,
                            arrival_airport,
                            last_name ,
                            first_name ,
                            seat,
                            boarding_time)::boarding_pass_record)
                    FROM flight f1
                    JOIN  boarding_pass bp ON f1.flight_id=bl.flight_id
                                AND bp.booking_leg_id=bl.booking_leg_id
                    JOIN passenger p ON p.passenger_id=bp.passenger_id)
             FROM booking_leg bl
             WHERE bl.booking_leg_id=p_booking_leg_id
;
END;
$body$ language plpgsql;
```

The issue with the above function is that nested composite types lose their structure in the resultant return so it is difficult to process and do much with the data. Internally the structure is still there however.

If function returns a user-defined type and that type is modified then any function returning that type must be dropped and recreated. 

### Functions and Security
* It can be useful to use the parameter **SECURITY DEFINER** as when we use this the tables and views used in the functions will be with the security permissions of the creator of the function.
* EXECUTE permission can then be given to a business user to use who does not have access to the underlying tables/views defined in the function.

### Stored Procedures

An alternative way to execute functions when calling them from another function or procedure:
* `PERFORM issue_boarding_pass(175820,462972, '22C', '2020-06-16 21:45'::timestamptz)`
    * This will execute the function but not return any results. Used instead of SELECT.

```
create or replace procedure transfer(
   sender int,
   receiver int,
   amount dec
)
language plpgsql
as $$
begin
    -- subtracting the amount from the sender's account
    update accounts
    set balance = balance - amount
    where id = sender;
    commit;
end;$$;
```

* Procedures do not return any values so no RETURN type is specified
* To execute a stored procedure you use:
    *  `CALL transfer(23,15,67)`

 ### Transaction Management
 * Procedures can commit or rollback a transaction within a procedure body whereas a function takes an all or nothing approach and is atomic.
     * A procedure can perform several commits whereas a function will only commit at the very end of execution.
     * Any **COMMIT** or **ROLLBACK** within the function body will terminate the current transaction and start a new one.
     * Bulk data load processing is a good usecase for this functionality.

### Sub-Blocks
* Can be used to logically group sets of statements within a larger block of code.
* Localizes variables so that will default their values every time the block is entered.
* Allows for targeted exception handling that makes sense for each block and can RAISE NOTICE about where an exception occurred.
* Able to reference variables in containing block using labels.
* BEGIN is not the same as Begin that starts a transaction.

```
CREATE or replace PROCEDURE multiple_blocks() 
AS
$mult$
<<outer_block>>
DECLARE
  counter integer := 0;
BEGIN 
   counter := counter + 1;
   RAISE NOTICE 'The current value of counter is %', counter;

   <<inner_block_1>>
   DECLARE 
       counter integer := 0;
   BEGIN 
       counter := counter + 10;
       RAISE NOTICE 'The current value of counter in the inner_block_1 is %', counter;
       RAISE NOTICE 'The current value of counter in the outer block is %', outer_block.counter;
   EXCEPTION WHEN OTHERS THEN
    RAISE NOTICE 'CASE#';
   END inner_block_1;

   <<inner_block_2>>
   DECLARE 
       counter integer := outer_block.counter;
   BEGIN 
       counter := counter + 10;
       RAISE NOTICE 'The current value of counter in the inner_block_2 is %', counter;
   EXCEPTION WHEN OTHERS THEN
    RAISE NOTICE 'CASE#2';
   END inner_block_2;

   RAISE NOTICE 'The current value of counter in the outer block is %', counter;
END outer_block
$mult$ LANGUAGE plpgsql;
```